# CNN

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime, timedelta
import re

def scrape_article_detail(url):
    try:
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
        response = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(response.text, "html.parser")

        title = soup.find('h1').get_text(strip=True) if soup.find('h1') else None

        # tanggal dari URL
        date = None
        match = re.search(r'/(\d{4})(\d{2})(\d{2})\d{6}', url)
        if match:
            year, month, day = match.groups()
            date = f"{day}-{month}-{year}"

        # kategori
        cat_match = re.search(r'cnnindonesia\.com/([^/]+)/', url)
        category = cat_match.group(1) if cat_match else "umum"

        # konten teks
        content = []
        container = soup.find('div', class_=re.compile(r'detail-text|article.*body|post.*content'))
        if container:
            for p in container.find_all('p'):
                txt = p.get_text(strip=True)
                if len(txt) > 30 and not any(x in txt.lower() for x in ['baca juga', 'googletag', 'simak video']):
                    content.append(txt)

        content_text = " ".join(content)
        if not title or len(content_text) < 150: return None

        return {
            "title": title,
            "date": date,
            "content": content_text,
            "category": category,
            "url": url,
            "source": "CNN Indonesia"
        }
    except:
        return None

def scrape_cnn_deep_scan(days_to_scrape=5, max_pages_per_day=15, delay=0.3):
    all_results = []
    global_processed_urls = set()
    today = datetime.now()

    for i in range(days_to_scrape):
        current_date = today - timedelta(days=i)
        date_str = current_date.strftime("%Y/%m/%d")

        for page in range(1, max_pages_per_day + 1):
            url_indeks = f"https://www.cnnindonesia.com/indeks?date={date_str}&page={page}"

            try:
                headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
                response = requests.get(url_indeks, headers=headers, timeout=20)
                soup = BeautifulSoup(response.text, "html.parser")

                links = []
                for a in soup.find_all('a', href=True):
                    href = a['href']
                    if 'cnnindonesia.com' in href and re.search(r'/\d{8,}', href):
                        if not any(x in href for x in ['/video/', '/foto/', '/tv/']):
                            links.append(href)

                links = list(set(links))

                if not links:
                    break

                new_on_page = 0
                for url in links:
                    if url not in global_processed_urls:
                        res = scrape_article_detail(url)
                        if res:
                            all_results.append(res)
                            global_processed_urls.add(url)
                            new_on_page += 1
                        time.sleep(delay)

                print(f"LOG: [{current_date.strftime('%d-%m-%Y')}] Halaman {page} | Berhasil mengambil {new_on_page} artikel")

            except Exception as e:
                print(f"ERROR: Terjadi kesalahan pada {date_str} hal {page}: {e}")
                break

        print(f"PROGRESS: Total akumulasi sementara: {len(all_results)} data")

    return all_results

if __name__ == "__main__":
    JUMLAH_HARI = 30
    MAX_HALAMAN = 20

    data_total = scrape_cnn_deep_scan(days_to_scrape=JUMLAH_HARI, max_pages_per_day=MAX_HALAMAN)

    if data_total:
        df_final = pd.DataFrame(data_total)
        nama_file = f"dataset_cnn.csv"
        df_final.to_csv(nama_file, index=False, encoding='utf-8-sig')
        print("-" * 60)
        print(f"STATUS: Proses selesai.")
        print(f"STATUS: Total data tersimpan: {len(df_final)}")
        print(f"STATUS: Nama file: {nama_file}")
        print("-" * 60)
    else:
        print("STATUS: Gagal mengambil data.")

LOG: [19-01-2026] Halaman 1 | Berhasil mengambil 10 artikel
LOG: [19-01-2026] Halaman 2 | Berhasil mengambil 7 artikel
LOG: [19-01-2026] Halaman 3 | Berhasil mengambil 9 artikel
LOG: [19-01-2026] Halaman 4 | Berhasil mengambil 8 artikel
LOG: [19-01-2026] Halaman 5 | Berhasil mengambil 9 artikel
LOG: [19-01-2026] Halaman 6 | Berhasil mengambil 8 artikel
LOG: [19-01-2026] Halaman 7 | Berhasil mengambil 10 artikel
LOG: [19-01-2026] Halaman 8 | Berhasil mengambil 9 artikel
LOG: [19-01-2026] Halaman 9 | Berhasil mengambil 10 artikel
LOG: [19-01-2026] Halaman 10 | Berhasil mengambil 8 artikel
PROGRESS: Total akumulasi sementara: 88 data
LOG: [18-01-2026] Halaman 1 | Berhasil mengambil 7 artikel
LOG: [18-01-2026] Halaman 2 | Berhasil mengambil 6 artikel
LOG: [18-01-2026] Halaman 3 | Berhasil mengambil 7 artikel
LOG: [18-01-2026] Halaman 4 | Berhasil mengambil 7 artikel
LOG: [18-01-2026] Halaman 5 | Berhasil mengambil 6 artikel
LOG: [18-01-2026] Halaman 6 | Berhasil mengambil 8 artikel
LOG: [1

In [ ]:
df = pd.read_csv("dataset_cnn.csv")
print(len(df))

4509


# Detik

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime, timedelta

def scrape_detik_detail(url):
    try:
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        title = soup.find('h1').get_text(strip=True) if soup.find('h1') else None

        container = soup.find('div', class_='detail__body-text') or \
                    soup.find('div', id='detikdetailtext') or \
                    soup.find('div', class_='detail_text') or \
                    soup.find('div', class_='itp_bodycontent')

        content = []
        if container:
            for s in container(['script', 'style', 'div', 'table', 'blockquote']):
                s.decompose()

            for p in container.find_all('p'):
                txt = p.get_text(strip=True)
                if len(txt) > 45 and not any(x in txt.lower() for x in ['baca juga', 'download aplikasi', 'simak video', 'foto:', 'detikcom']):
                    content.append(txt)

        content_text = " ".join(content)
        if not title or len(content_text) < 300: return None

        return {
            "title": title,
            "content": content_text,
            "url": url,
            "source": "Detik.com"
        }
    except:
        return None

def scrape_detik_collection(target_total=4050, days_back=150):
    all_results = []
    processed_urls = set()
    today = datetime.now()
    channels = ['news', 'finance', 'hot', 'techno', 'edu', 'sport', 'travel', 'food']

    for i in range(days_back):
        if len(all_results) >= target_total: break

        current_date = today - timedelta(days=i)
        date_str = current_date.strftime("%m/%d/%Y")

        for cat in channels:
            if len(all_results) >= target_total: break

            for page in range(1, 16):
                url_indeks = f"https://{cat}.detik.com/indeks?date={date_str}&page={page}"

                try:
                    headers = {"User-Agent": "Mozilla/5.0"}
                    response = requests.get(url_indeks, headers=headers, timeout=15)
                    soup = BeautifulSoup(response.text, "html.parser")

                    articles = soup.find_all('article')
                    if not articles: break

                    new_count = 0
                    for item in articles:
                        a_tag = item.find('a', href=True)
                        if a_tag and 'detik.com' in a_tag['href']:
                            url = a_tag['href']
                            if url not in processed_urls:
                                res = scrape_detik_detail(url)
                                if res:
                                    res['date'] = current_date.strftime("%d-%m-%Y")
                                    res['category'] = cat
                                    all_results.append(res)
                                    processed_urls.add(url)
                                    new_count += 1

                        if len(all_results) >= target_total: break

                    if new_count == 0: break

                    if len(all_results) % 50 == 0 and new_count > 0:
                        print(f"[{current_date.strftime('%d-%m-%Y')}] Akumulasi data: {len(all_results)}")

                except Exception:
                    break

            time.sleep(0.1)

    return all_results

if __name__ == "__main__":
    start_time = time.time()

    final_data = scrape_detik_collection(target_total=4000, days_back=180)
    df = pd.DataFrame(final_data)

    if not df.empty:
        df['label'] = 0
        nama_file = f"detik_dataset_{len(df)}.csv"
        df.to_csv(nama_file, index=False, encoding='utf-8-sig')

        duration = (time.time() - start_time) / 60
        print("-" * 60)
        print(f"Proses selesai. Total data tersimpan: {len(df)}")
        print(f"Nama file: {nama_file}")
        print("-" * 60)

[09-01-2026] Akumulasi data: 1950
[02-01-2026] Akumulasi data: 3500
[31-12-2025] Akumulasi data: 3950
[31-12-2025] Akumulasi data: 4000
------------------------------------------------------------
Proses selesai. Total data tersimpan: 4003
Nama file: detik_dataset_4003.csv
------------------------------------------------------------


# Kompas

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime, timedelta
import random


def scrape_kompas_article_detail(url):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        }

        if '?page=all' not in url:
            full_url = url + '?page=all'
        else:
            full_url = url

        response = requests.get(full_url, headers=headers, timeout=10)

        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, "html.parser")

        title = None
        h1 = soup.find('h1', class_='read__title')
        if h1:
            title = h1.get_text(strip=True)
        elif soup.find('h1'):
            title = soup.find('h1').get_text(strip=True)

        content = []
        container = soup.find('div', class_='read__content')

        if container:
            for p in container.find_all('p'):
                txt = p.get_text(strip=True)
                if len(txt) > 45:
                    lower = txt.lower()
                    if not any(x in lower for x in [
                        'baca juga', 'lihat juga', 'simak', 'editor:',
                        'reporter:', 'kompas.com', 'foto:', 'video:',
                        'advertisement', 'dapatkan update'
                    ]):
                        content.append(txt)

        content_text = " ".join(content)

        if not title or len(content_text) < 300:
            return None

        return {
            "title": title,
            "content": content_text,
            "url": url,
            "source": "Kompas.com"
        }
    except:
        return None


def scrape_kompas_index_by_date(date_obj, page=1):
    article_urls = []

    date_str = date_obj.strftime("%Y-%m-%d")

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    url_index = f"https://indeks.kompas.com/?site=all&date={date_str}&page={page}"

    try:
        response = requests.get(url_index, headers=headers, timeout=15)

        if response.status_code != 200:
            return []

        soup = BeautifulSoup(response.text, "html.parser")

        articles = soup.find_all('article')
        for article in articles:
            a_tag = article.find('a', href=True)
            if a_tag:
                href = a_tag['href']
                if 'kompas.com' in href and '/read/' in href:
                    article_urls.append(href)

        links = soup.find_all('a', class_=['article__link', 'artikel__link'])
        for link in links:
            href = link.get('href', '')
            if 'kompas.com' in href and '/read/' in href:
                article_urls.append(href)

        all_links = soup.find_all('a', href=True)
        for link in all_links:
            href = link['href']
            if '/read/' in href and 'kompas.com' in href:
                if href.startswith('http'):
                    article_urls.append(href)
                elif href.startswith('/'):
                    article_urls.append(f"https://www.kompas.com{href}")

    except:
        pass

    return list(set(article_urls))


def collect_kompas_articles(target_total=4000, days_back=150):
    all_results = []
    processed_urls = set()
    today = datetime.now()

    consecutive_empty = 0

    for day_idx in range(days_back):
        if len(all_results) >= target_total:
            break

        if consecutive_empty > 7:
            print("Process stopped due to consecutive empty dates")
            break

        current_date = today - timedelta(days=day_idx)
        date_str = current_date.strftime('%d-%m-%Y')

        print(f"\nDate: {date_str} | Collected: {len(all_results)}")

        day_count = 0

        for page in range(1, 16):
            if len(all_results) >= target_total:
                break

            article_urls = scrape_kompas_index_by_date(current_date, page)

            if not article_urls:
                break

            print(f"  Page {page}: {len(article_urls)} URLs", end=" | ")

            new_count = 0
            for url in article_urls:
                if url in processed_urls:
                    continue

                if len(all_results) >= target_total:
                    break

                detail = scrape_kompas_article_detail(url)

                if detail:
                    detail['date'] = date_str
                    all_results.append(detail)
                    processed_urls.add(url)
                    new_count += 1
                    day_count += 1
                else:
                    processed_urls.add(url)

                time.sleep(0.05)

            print(f"New articles: {new_count}")

            if new_count == 0:
                break

            time.sleep(0.1)

        consecutive_empty = 0 if day_count > 0 else consecutive_empty + 1

        if len(all_results) % 500 == 0 and len(all_results) > 0:
            df_temp = pd.DataFrame(all_results)
            checkpoint_file = f"kompas_checkpoint_{len(all_results)}.csv"
            df_temp.to_csv(checkpoint_file, index=False, encoding='utf-8-sig')
            print(f"Checkpoint saved: {checkpoint_file}")

    return all_results


if __name__ == "__main__":
    start_time = time.time()


    final_data = collect_kompas_articles(target_total=4000, days_back=180)

    df = pd.DataFrame(final_data)

    if not df.empty:
        df['label'] = 0

        timestamp = datetime.now().strftime("%Y%m%d_%H%M")
        filename = f"dataset_kompas_{len(df)}_{timestamp}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')

        duration = (time.time() - start_time) / 60

        print("\Proses selesai")
        print(f"Total articles : {len(df)}")
        print(f"Output file    : {filename}")
        print(df[['title', 'source', 'date']].head(10))
    else:
        print("No data collected")
        print("Possible causes: request blocking or page structure changes")


<>:201: SyntaxWarning: invalid escape sequence '\P'
<>:201: SyntaxWarning: invalid escape sequence '\P'
/tmp/ipython-input-366799437.py:201: SyntaxWarning: invalid escape sequence '\P'
  print("\Proses selesai")



Date: 20-01-2026 | Collected: 0
  Page 1: 21 URLs | New articles: 21
  Page 2: 20 URLs | New articles: 19
  Page 3: 19 URLs | New articles: 16
  Page 4: 20 URLs | New articles: 19
  Page 5: 21 URLs | New articles: 19
  Page 6: 20 URLs | New articles: 19
  Page 7: 21 URLs | New articles: 20
  Page 8: 21 URLs | New articles: 20
  Page 9: 21 URLs | New articles: 20
  Page 10: 20 URLs | New articles: 19
  Page 11: 21 URLs | New articles: 18
  Page 12: 21 URLs | New articles: 20
  Page 13: 19 URLs | New articles: 18
  Page 14: 21 URLs | New articles: 20
  Page 15: 20 URLs | New articles: 19

Date: 19-01-2026 | Collected: 287
  Page 1: 21 URLs | New articles: 20
  Page 2: 20 URLs | New articles: 19
  Page 3: 20 URLs | New articles: 19
  Page 4: 21 URLs | New articles: 20
  Page 5: 19 URLs | New articles: 17
  Page 6: 21 URLs | New articles: 20
  Page 7: 20 URLs | New articles: 19
  Page 8: 20 URLs | New articles: 19
  Page 9: 21 URLs | New articles: 20
  Page 10: 20 URLs | New articles: 19


# TurnbackHoax

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from datetime import datetime
import os
import glob

def scrape_article_detail(url, session):
    """Ekstrak detail artikel dengan konten MAKSIMAL"""
    try:
        response = session.get(url, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # Ekstrak judul
        title_tag = soup.find('h1')
        title = title_tag.get_text(strip=True) if title_tag else None

        # Ekstrak tanggal
        date = None
        for elem in soup.find_all(['div', 'span', 'time'], class_=re.compile(r'text-sm|date|time|published')):
            text = elem.get_text(strip=True)
            match = re.search(r'(\d{2}/\d{2}/\d{4})', text)
            if match:
                date = match.group(1)
                break

        if not date:
            meta_date = soup.find('meta', {'property': 'article:published_time'})
            if meta_date and meta_date.get('content'):
                try:
                    dt = datetime.fromisoformat(meta_date['content'].replace('Z', '+00:00'))
                    date = dt.strftime('%d/%m/%Y')
                except:
                    pass

        if not date:
            page_text = soup.get_text()
            matches = re.findall(r'(\d{2}/\d{2}/\d{4})', page_text)
            for match in matches:
                parts = match.split('/')
                if len(parts) == 3:
                    day, month, year = int(parts[0]), int(parts[1]), int(parts[2])
                    if 1 <= day <= 31 and 1 <= month <= 12 and 2015 <= year <= 2026:
                        date = match
                        break

        # Ekstrak kategori
        category = "Uncategorized"
        category_link = soup.find('a', href=re.compile(r'category='))
        if category_link:
            category = category_link.get_text(strip=True)

        #  EKSTRAKSI KONTEN
        content_parts = []

        # Cari container artikel utama
        main_content = None
        possible_containers = soup.find_all(['div', 'article', 'section'],
            class_=re.compile(r'article|content|post|entry|main|body', re.I))

        for container in possible_containers:
            text = container.get_text(strip=True)
            if len(text) > 300:
                main_content = container
                break

        if not main_content:
            main_content = soup.find('body')
        if not main_content:
            main_content = soup

        paragraphs = main_content.find_all('p')

        # Blacklist navigasi/footer
        blacklist_phrases = [
            'menu lainnya', 'cari', 'profil tim', 'artikel terbaru',
            'affiliation', 'tautan cepat', 'hubungi kami', 'whatsapp hotline',
            'kalimasada', 'epicentrum', 'kuningan', 'jakarta selatan',
            'masyarakat anti fitnah', 'mafindo', 'turnbackhoax',
            'load more', 'lihat semua', 'kembali ke atas',
            'bagikan artikel', 'share this', 'follow us',
            'copyright ©', 'all rights reserved', '© 20',
            'privacy policy', 'terms of service', 'cookie policy'
        ]

        # Whitelist konten artikel hoax
        whitelist_patterns = [
            r'salah|benar|hoax|penipuan|misleading',
            r'narasi|klaim|beredar|viral',
            r'sumber:|referensi:|faktanya:',
            r'berdasarkan|dilansir|dikutip',
            r'kesimpulan|hasil penelusuran',
        ]

        for p in paragraphs:
            text = p.get_text(strip=True)

            if not text or len(text) < 10:
                continue

            text_lower = text.lower()

            if any(phrase in text_lower for phrase in blacklist_phrases):
                continue

            if text.startswith('http') and ' ' not in text and len(text) > 50:
                continue

            is_article_content = any(re.search(pattern, text_lower) for pattern in whitelist_patterns)

            if is_article_content or (len(text) > 30 and not text.lower().startswith('http')):
                content_parts.append(text)

        # Fallback: cari di div lain kalau konten kurang
        if len(' '.join(content_parts)) < 200:
            all_divs = main_content.find_all(['div', 'section'])
            for div in all_divs:
                if div.find('p'):
                    continue

                text = div.get_text(strip=True)
                if len(text) > 50 and len(text) < 500:
                    text_lower = text.lower()
                    if not any(phrase in text_lower for phrase in blacklist_phrases):
                        if any(re.search(pattern, text_lower) for pattern in whitelist_patterns):
                            content_parts.append(text)

        content = " ".join(content_parts)
        content = re.sub(r'\s+', ' ', content).strip()

        if not title:
            return None

        if len(content) < 50:
            return None

        return {
            "title": title,
            "date": date if date else "N/A",
            "category": category,
            "content": content,
            "url": url,
            "source": "TurnBackHoax.id"
        }

    except Exception as e:
        return None

def scrape_article_list(page_number, session):
    """Ambil daftar URL artikel dari halaman index"""
    try:
        url = f"https://turnbackhoax.id/articles?page={page_number}"
        response = session.get(url, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        article_urls = set()
        for link in soup.find_all('a', href=True):
            href = link['href']
            if '/articles/' in href and re.search(r'/articles/\d+', href):
                if href.startswith('http'):
                    full_url = href
                elif href.startswith('/'):
                    full_url = f"https://turnbackhoax.id{href}"
                else:
                    full_url = f"https://turnbackhoax.id/{href}"
                article_urls.add(full_url)

        return list(article_urls)

    except Exception as e:
        return []

def load_existing_data(backup_dir="backups"):
    """Load data dari backup terakhir (untuk resume)"""
    if not os.path.exists(backup_dir):
        return [], set()

    backup_files = glob.glob(f"{backup_dir}/backup_*.csv")

    if not backup_files:
        return [], set()

    # Ambil backup terakhir (file terbesar/terbaru)
    latest_backup = max(backup_files, key=os.path.getctime)

    print(f"Menemukan backup: {latest_backup}")

    try:
        df = pd.read_csv(latest_backup, encoding='utf-8-sig')
        existing_urls = set(df['url'].tolist())
        existing_data = df.to_dict('records')

        print(f" Loaded {len(existing_data)} artikel dari backup")
        return existing_data, existing_urls
    except Exception as e:
        print(f"  Error loading backup: {e}")
        return [], set()

def save_backup(data, backup_dir="backups", prefix="backup"):
    """Simpan backup dengan timestamp"""
    if not data:
        return

    # Buat folder backup kalau belum ada
    os.makedirs(backup_dir, exist_ok=True)

    # Nama file dengan jumlah artikel dan timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{backup_dir}/{prefix}_{len(data)}_{timestamp}.csv"

    df = pd.DataFrame(data)
    df.to_csv(filename, index=False, encoding='utf-8-sig')

    print(f"   Backup: {filename}")

    # Hapus backup lama (keep 5 terakhir)
    backup_files = sorted(glob.glob(f"{backup_dir}/{prefix}_*.csv"),
                         key=os.path.getctime, reverse=True)

    if len(backup_files) > 5:
        for old_file in backup_files[5:]:
            try:
                os.remove(old_file)
                print(f"    Hapus backup lama: {old_file}")
            except:
                pass

def scrape_turnbackhoax(target_articles=13000, max_pages=6000, delay=0.5,
                        resume=True, backup_interval=500):
    """
    Scraping dengan fitur backup otomatis dan resume

    Args:
        target_articles: Target jumlah artikel
        max_pages: Maksimal halaman yang di-scrape
        delay: Delay antar request (detik)
        resume: Load dari backup terakhir (True/False)
        backup_interval: Simpan backup setiap N artikel
    """

    # Load data existing kalau resume=True
    if resume:
        all_articles, processed_urls = load_existing_data()
        if all_articles:
            print(f" RESUME: Melanjutkan dari {len(all_articles)} artikel\n")
    else:
        all_articles = []
        processed_urls = set()
        print(" Mulai dari awal\n")

    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    })

    print("=" * 70)
    print("SCRAPING TURNBACKHOAX.ID - WITH AUTO BACKUP")
    print("=" * 70)
    print(f"Target: {target_articles} artikel")
    print(f"Sudah ada: {len(all_articles)} artikel")
    print(f"Butuh: {max(0, target_articles - len(all_articles))} artikel lagi")
    print(f"Backup interval: setiap {backup_interval} artikel")
    print(f"Max pages: {max_pages}")
    print(f"Delay: {delay}s\n")

    page = 1
    empty_count = 0
    last_backup_count = len(all_articles)

    try:
        while len(all_articles) < target_articles and page <= max_pages:
            print(f"[Page {page:4d}] ", end="", flush=True)

            urls = scrape_article_list(page, session)

            if not urls:
                empty_count += 1
                print(f"Kosong (streak: {empty_count})")
                if empty_count >= 5:
                    print("\n  5 halaman kosong berturut-turut. Berhenti.")
                    break
                page += 1
                time.sleep(delay)
                continue

            empty_count = 0
            new_urls = [u for u in urls if u not in processed_urls]

            if not new_urls:
                print("Semua URL sudah diproses")
                page += 1
                time.sleep(delay)
                continue

            print(f"{len(new_urls)} baru → ", end="", flush=True)

            success = 0
            failed = 0
            for url in new_urls:
                if len(all_articles) >= target_articles:
                    break

                article_data = scrape_article_detail(url, session)
                if article_data:
                    all_articles.append(article_data)
                    processed_urls.add(url)
                    success += 1
                else:
                    failed += 1

                time.sleep(delay)

                # Auto backup setiap N artikel
                if len(all_articles) - last_backup_count >= backup_interval:
                    print(f"\n Auto-saving backup...")
                    save_backup(all_articles)
                    last_backup_count = len(all_articles)
                    print(f"[Page {page:4d}] ", end="", flush=True)

            print(f"✓ {success} OK, ✗ {failed} Skip | Total: {len(all_articles)}")

            page += 1
            time.sleep(delay)

    except KeyboardInterrupt:
        print("\n\n  Scraping dihentikan oleh user (Ctrl+C)")
        print(" Menyimpan backup terakhir...")
        save_backup(all_articles, prefix="interrupted")
        print("Backup tersimpan!")

    except Exception as e:
        print(f"\n\nError: {e}")
        print(" Menyimpan backup darurat...")
        save_backup(all_articles, prefix="emergency")
        print(" Backup darurat tersimpan!")

    return all_articles

def save_to_csv(data, filename="dataset_turnbackhoax_final.csv"):
    """Simpan dataset final dengan statistik lengkap"""
    if not data:
        print("\n Tidak ada data untuk disimpan")
        return

    df = pd.DataFrame(data)

    # Hapus duplikat
    before = len(df)
    df = df.drop_duplicates(subset=['url'], keep='first')
    if before > len(df):
        print(f"\n  Duplikat dihapus: {before - len(df)}")

    # Sort by date
    try:
        df['date_sort'] = pd.to_datetime(df['date'], format='%d/%m/%Y', errors='coerce')
        df = df.sort_values('date_sort', ascending=False, na_position='last')
        df = df.drop('date_sort', axis=1)
    except:
        pass

    # Simpan
    df.to_csv(filename, index=False, encoding='utf-8-sig')

    print("\n" + "=" * 70)
    print("DATASET FINAL TERSIMPAN!")
    print("=" * 70)
    print(f"Total artikel: {len(df)}")
    print(f"File output: {filename}")

    # Statistik kategori
    print(f"\n📊 Top 10 Kategori:")
    for cat, count in df['category'].value_counts().head(10).items():
        print(f"  {cat:20s}: {count:4d} artikel ({count/len(df)*100:.1f}%)")

    # Statistik tanggal
    valid_dates = (df['date'] != 'N/A').sum()
    print(f"\n Tanggal valid: {valid_dates}/{len(df)} ({valid_dates/len(df)*100:.1f}%)")

    # STATISTIK KONTEN
    df['content_length'] = df['content'].str.len()
    print(f"\n STATISTIK PANJANG KONTEN:")
    print(f"  Rata-rata : {df['content_length'].mean():.0f} karakter")
    print(f"  Median    : {df['content_length'].median():.0f} karakter")
    print(f"  Q1 (25%)  : {df['content_length'].quantile(0.25):.0f} karakter")
    print(f"  Q3 (75%)  : {df['content_length'].quantile(0.75):.0f} karakter")
    print(f"  Min       : {df['content_length'].min()} karakter")
    print(f"  Max       : {df['content_length'].max()} karakter")

    # Distribusi panjang konten
    print(f"\nDistribusi Panjang Konten:")
    bins = [
        (0, 100, "Sangat Pendek"),
        (100, 300, "Pendek"),
        (300, 600, "Sedang"),
        (600, 1000, "Cukup Panjang"),
        (1000, 2000, "Panjang"),
        (2000, float('inf'), "Sangat Panjang")
    ]

    for min_len, max_len, label in bins:
        if max_len == float('inf'):
            count = (df['content_length'] >= min_len).sum()
            print(f"  {label:15s} (≥{min_len:4d}): {count:5d} artikel ({count/len(df)*100:.1f}%)")
        else:
            count = ((df['content_length'] >= min_len) & (df['content_length'] < max_len)).sum()
            print(f"  {label:15s} ({min_len:4d}-{max_len:4d}): {count:5d} artikel ({count/len(df)*100:.1f}%)")

    # Sample konten (5 artikel random)
    print(f"\n Sample Konten (5 artikel random):")
    for idx, row in df.sample(min(5, len(df))).iterrows():
        print(f"\n  [{row['category']}] {row['title'][:60]}...")
        print(f"  Konten ({len(row['content'])} char): {row['content'][:150]}...")

    print("\n" + "=" * 70)
    print("Dataset siap untuk skripsi!")
    print("=" * 70)

if __name__ == "__main__":
    #  KONFIGURASI
    TARGET = 13000           # Target artikel
    MAX_PAGES = 6000         # Max halaman
    DELAY = 0.5              # Delay antar request (detik)
    RESUME = True            # True = lanjut dari backup, False = mulai baru
    BACKUP_INTERVAL = 500    # Backup setiap N artikel

    print("\n Memulai scraping...\n")

    data = scrape_turnbackhoax(
        target_articles=TARGET,
        max_pages=MAX_PAGES,
        delay=DELAY,
        resume=RESUME,
        backup_interval=BACKUP_INTERVAL
    )

    if data:
        save_to_csv(data, "dataset_turnbackhoax_final.csv")
        print(f"\nBERHASIL! {len(data)} artikel tersimpan")
    else:
        print("\nGAGAL! Tidak ada data yang berhasil dikumpulkan")


🚀 Memulai scraping...

SCRAPING TURNBACKHOAX.ID - WITH AUTO BACKUP
Target: 13000 artikel
Sudah ada: 0 artikel
Butuh: 13000 artikel lagi
Backup interval: setiap 500 artikel
Max pages: 6000
Delay: 0.5s

[Page    1] 10 baru → ✓ 10 OK, ✗ 0 Skip | Total: 10
[Page    2] 10 baru → ✓ 10 OK, ✗ 0 Skip | Total: 20
[Page    3] 10 baru → ✓ 10 OK, ✗ 0 Skip | Total: 30
[Page    4] 10 baru → ✓ 10 OK, ✗ 0 Skip | Total: 40
[Page    5] 10 baru → ✓ 10 OK, ✗ 0 Skip | Total: 50
[Page    6] 10 baru → ✓ 10 OK, ✗ 0 Skip | Total: 60
[Page    7] 10 baru → ✓ 10 OK, ✗ 0 Skip | Total: 70
[Page    8] 10 baru → ✓ 10 OK, ✗ 0 Skip | Total: 80
[Page    9] 10 baru → ✓ 10 OK, ✗ 0 Skip | Total: 90
[Page   10] 10 baru → ✓ 10 OK, ✗ 0 Skip | Total: 100
[Page   11] 10 baru → ✓ 10 OK, ✗ 0 Skip | Total: 110
[Page   12] 10 baru → ✓ 10 OK, ✗ 0 Skip | Total: 120
[Page   13] 10 baru → ✓ 10 OK, ✗ 0 Skip | Total: 130
[Page   14] 10 baru → ✓ 10 OK, ✗ 0 Skip | Total: 140
[Page   15] 10 baru → ✓ 10 OK, ✗ 0 Skip | Total: 150
[Page   16] 

In [ ]:
from google.colab import files

files.download("dataset_turnbackhoax.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>